# kasa: training the Twi voice on Colab

Runs the whole pipeline on a GPU: data preparation, fine-tuning, checkpoint
scoring and sample generation. Everything is written to Google Drive, so a
disconnect doesn't lose the run. Re-running the notebook resumes where it stopped.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and keep outputs on Drive

In [ ]:
REPO = "https://github.com/YOUR-USERNAME/twi-kasa.git"   # <- your GitHub repo
SPEAKER = "2"                                           # WAXAL speaker to train on

from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/kasa"
os.makedirs(f"{DRIVE}/runs", exist_ok=True)
os.makedirs(f"{DRIVE}/data", exist_ok=True)

if not os.path.exists("/content/twi-kasa"):
    !git clone $REPO /content/twi-kasa
%cd /content/twi-kasa
!git pull -q

# runs/ and data/ live on Drive; the repo just points at them
for d in ("runs", "data/processed", "data/raw"):
    target = f"{DRIVE}/{d.replace('/', '_')}"
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(target, d)

## 2. Install

In [ ]:
!pip install -q -r requirements.txt Cython tensorboard
!python src/check_setup.py

## 3. Prepare the data

Downloads WAXAL Twi TTS, force-aligns every recording and cuts it into
sentence clips of at most 15 s, for one speaker. On a T4 this takes a few minutes.

In [ ]:
if not os.path.exists("data/processed/train/metadata.csv"):
    !python src/prepare_data.py --speakers $SPEAKER

import pandas as pd
for split in ("train", "validation", "test"):
    m = pd.read_csv(f"data/processed/{split}/metadata.csv")
    print(f"{split:<10} {len(m):>5} clips  {m.duration.sum()/3600:.2f} h  "
          f"median score {m.score.median():.3f}")

## 4. Fine-tune

Fine-tunes `facebook/mms-tts-aka` with the text encoder and duration predictor
frozen (see `configs/train.json`). Checkpoints are saved every 500 steps to Drive.
If Colab disconnects, run this cell again: the harness resumes from the last checkpoint.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/kasa-twi

In [ ]:
!python src/train.py

## 5. Score every checkpoint and keep the best

Each checkpoint reads the held-out test sentences aloud. The Akan speech
recogniser transcribes them, and we compare CER with the base model and with
the real recordings.

In [ ]:
import glob, json
LIMIT = 60
runs = {"real": None, "base": "facebook/mms-tts-aka"}
runs.update({os.path.basename(p): p for p in sorted(glob.glob("runs/kasa-twi/exports/step-*"),
                                                      key=lambda p: int(p.split("-")[-1]))})
for name, model in runs.items():
    if os.path.exists(f"outputs/eval/{name}/summary.json"):
        continue
    if model:
        !python src/score.py data/processed/test --model $model --name $name --limit $LIMIT
    else:
        !python src/score.py data/processed/test --name real --limit $LIMIT

results = {n: json.load(open(f"outputs/eval/{n}/summary.json")) for n in runs}
table = pd.DataFrame(results).T[["cer", "wer", "open_vowels_kept", "open_vowels_flattened"]]
print(table)

In [ ]:
steps = [n for n in results if n.startswith("step-")]
best = min(steps, key=lambda n: results[n]["cer"])
print("best checkpoint:", best, results[best])

!rm -rf checkpoints/kasa-twi
!cp -r runs/kasa-twi/exports/$best checkpoints/kasa-twi
!cp -r checkpoints/kasa-twi $DRIVE/kasa-twi-best
!cp -r outputs $DRIVE/

## 6. Speak unseen sentences

In [ ]:
!python src/speak.py --file data/external/unseen_sentences.txt --model checkpoints/kasa-twi --out-dir outputs/samples/kasa
!python src/speak.py --file data/external/unseen_sentences.txt --model facebook/mms-tts-aka --out-dir outputs/samples/base
!cp -r outputs/samples $DRIVE/

from IPython.display import Audio, display
for line in open("outputs/samples/kasa/prompts.tsv", encoding="utf-8"):
    name, text = line.rstrip("\n").split("\t")
    print(text)
    display(Audio(f"outputs/samples/kasa/{name}"))